# 🌊 GridForecast — 07: Drift Detection

**Цел:** Откриване на drift в data и model predictions.

**Защо е важно:**
- Consumption patterns се променят с времето
- Temperature trends се shift-ват
- Holidays се променят
- Моделът трябва да се retrain-ва периодично

**Методи:**
1. **ADWIN** — Adaptive Windowing (концептен drift)
2. **PSI** — Population Stability Index (data drift)
3. **KS test** — Kolmogorov-Smirnov (distribution shift)
4. **Residual monitoring** — model drift

**Структура:**
1. Setup
2. Load data + predictions
3. PSI (Population Stability Index)
4. KS test (Kolmogorov-Smirnov)
5. ADWIN (Adaptive Windowing)
6. Residual monitoring
7. Feature drift analysis
8. Alert system
9. Визуализации
10. Extended analysis
11. **Fix 1-3: Seasonal PSI + pattern comparison + weighted verdict**
12. **Финал: запис на `drift_summary.parquet`**

## 1️⃣ Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.spatial.distance import jensenshannon

from gridforecast.config import (
    SYNTHETIC_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    TRAIN_RATIO,
    RANDOM_SEED,
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titleweight"] = "bold"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_SEED)

print("✅ Imports OK")

✅ Imports OK


## 2️⃣ Load data + predictions

⚠️ **Важно:** `boosting_stable_predictions.parquet` трябва да съществува в `models/`.
Ако е преместен в `_backup_YYYYMMDD_HHMMSS/`, notebook-ът ще хвърли **ясна грешка** с инструкции.

In [2]:
# Load data
df = pd.read_parquet(SYNTHETIC_DIR / "consumption_stable.parquet")
df.index = pd.DatetimeIndex(df.index)

split_idx = int(TRAIN_RATIO * len(df))
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

print(f"Data shape: {df.shape}")
print(f"Train: {len(train):,} | Test: {len(test):,}")

# Load predictions — с fallback логика
pred_path = MODELS_DIR / "boosting_stable_predictions.parquet"
if not pred_path.exists():
    raise FileNotFoundError(
        f"\n❌ Липсва: {pred_path}\n\n"
        f"Опции:\n"
        f"  A) Върни файла от backup:\n"
        f"     cp models/_backup_*/boosting_stable_predictions.parquet models/\n\n"
        f"  B) Regenerate predictions от production_model.pkl (виж 08_retraining.ipynb)\n"
    )

pred_df = pd.read_parquet(pred_path)
print(f"\nPredictions shape: {pred_df.shape}")
print(f"Columns: {pred_df.columns.tolist()}")
pred_df.head()

Data shape: (17520, 9)
Train: 14,016 | Test: 3,504

Predictions shape: (3504, 6)
Columns: ['timestamp', 'actual', 'boosting_best', 'xgb', 'lgb', 'snaive']


,timestamp,actual,boosting_best,xgb,lgb,snaive
0,2024-08-07 00:00:00,1129.031925,1121.573237,1121.573237,1121.573237,NaN
1,2024-08-07 01:00:00,1158.411338,1107.516143,1107.516143,1107.516143,NaN
2,2024-08-07 02:00:00,1078.458164,1105.107250,1105.107250,1105.107250,NaN
3,2024-08-07 03:00:00,1102.592650,1090.774612,1090.774612,1090.774612,NaN
4,2024-08-07 04:00:00,1061.297865,1105.890781,1105.890781,1105.890781,NaN


## 3️⃣ PSI — Population Stability Index

**PSI** измерва колко се е променило разпределението между две групи (train vs test).

**Interpretation:**
- PSI < 0.1 → **No drift** ✅
- 0.1 ≤ PSI < 0.25 → **Moderate drift** ⚠️
- PSI ≥ 0.25 → **Significant drift** 🚨 (retrain!)

⚠️ **Внимание:** Full PSI може да е **false positive** при seasonal data (train = всички сезони, test = само есен/зима). Виж Fix 1 по-долу.

In [3]:
def psi(expected, actual, buckets=10):
    """Population Stability Index."""
    expected = np.asarray(expected)
    actual = np.asarray(actual)

    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints)

    exp_counts, _ = np.histogram(expected, bins=breakpoints)
    act_counts, _ = np.histogram(actual, bins=breakpoints)

    exp_pct = (exp_counts + 0.0001) / (len(expected) + 0.0001 * len(breakpoints))
    act_pct = (act_counts + 0.0001) / (len(actual) + 0.0001 * len(breakpoints))

    psi_values = (act_pct - exp_pct) * np.log(act_pct / exp_pct)
    return np.sum(psi_values)


print("=" * 70)
print("PSI — Population Stability Index")
print("=" * 70)

feature_cols = ["consumption_mw", "temperature_c"]

psi_results = {}
for col in feature_cols:
    psi_val = psi(train[col].values, test[col].values)
    psi_results[col] = psi_val

    if psi_val < 0.1:
        status = "✅ No drift"
    elif psi_val < 0.25:
        status = "⚠️ Moderate drift"
    else:
        status = "🚨 Significant drift"
    print(f"  {col:20s}: PSI = {psi_val:.4f}  {status}")

if "actual" in pred_df.columns and "boosting_best" in pred_df.columns:
    psi_pred = psi(pred_df["actual"].values, pred_df["boosting_best"].values)
    print(f"\n  Predictions vs Actual: PSI = {psi_pred:.4f}")
    psi_results["predictions"] = psi_pred

PSI — Population Stability Index
  consumption_mw      : PSI = 0.5318  🚨 Significant drift
  temperature_c       : PSI = 0.0883  ✅ No drift

  Predictions vs Actual: PSI = 0.0080


## 4️⃣ KS test — Kolmogorov-Smirnov

**KS test** сравнява две distributions. **p < 0.05 → drift.**

⚠️ **Внимание:** KS е много sensitive при големи samples — малки разлики дават p ≈ 0. Виж Fix 1.

In [4]:
print("=" * 70)
print("KS TEST — Kolmogorov-Smirnov")
print("=" * 70)

ks_results = {}
for col in feature_cols:
    ks_stat, ks_p = stats.ks_2samp(train[col].values, test[col].values)
    ks_results[col] = {"stat": ks_stat, "p": ks_p}
    # ЗАБЕЛЕЖКА: p < 0.05 → DRIFT (не OK!)
    status = "🚨 Drift" if ks_p < 0.05 else "✅ No drift"
    print(f"  {col:20s}: KS = {ks_stat:.4f}, p = {ks_p:.4e}  {status}")

print("\nInterpretation:")
print("  p < 0.05 → drift detected (distributions differ)")
print("  p ≥ 0.05 → no significant difference")

KS TEST — Kolmogorov-Smirnov
  consumption_mw      : KS = 0.2579, p = 6.2450e-165  🚨 Drift
  temperature_c       : KS = 0.0926, p = 2.2669e-21  🚨 Drift

Interpretation:
  p < 0.05 → drift detected (distributions differ)
  p ≥ 0.05 → no significant difference


## 5️⃣ ADWIN — Adaptive Windowing

**ADWIN** открива концептен drift в стрийм от данни (residuals).

⚠️ **Внимание:** ADWIN е много sensitive при seasonal residuals → false positives. Виж Fix 3.

In [5]:
def adwin_detect(data, delta=0.01, max_window=1000):
    """Опростен ADWIN — връща drift indices."""
    drift_points = []
    n = len(data)

    for t in range(max_window, n, max_window // 4):
        window = data[max(0, t - max_window):t]
        mid = len(window) // 2

        if mid < 10:
            continue

        w1, w2 = window[:mid], window[mid:]
        m = 1.0 / (1.0 / len(w1) + 1.0 / len(w2))
        eps = np.sqrt((1.0 / (2 * m)) * np.log(4.0 / delta))

        if abs(w1.mean() - w2.mean()) > eps:
            drift_points.append(t)

    return drift_points


print("=" * 70)
print("ADWIN — Adaptive Windowing")
print("=" * 70)

drift_points = []
if "actual" in pred_df.columns and "boosting_best" in pred_df.columns:
    residuals = pred_df["actual"].values - pred_df["boosting_best"].values
    drift_points = adwin_detect(residuals, delta=0.01, max_window=500)

    print(f"\nОткрити drift points: {len(drift_points)}")
    if len(drift_points) > 0:
        print("Timestamps (първите 10):")
        for idx in drift_points[:10]:
            if idx < len(pred_df):
                print(f"  {pred_df['timestamp'].iloc[idx]}")
    else:
        print("✅ Не са открити drift points в residuals")

ADWIN — Adaptive Windowing

Открити drift points: 25
Timestamps (първите 10):
  2024-08-27 20:00:00
  2024-09-02 01:00:00
  2024-09-07 06:00:00
  2024-09-12 11:00:00
  2024-09-17 16:00:00
  2024-09-22 21:00:00
  2024-09-28 02:00:00
  2024-10-03 07:00:00
  2024-10-08 12:00:00
  2024-10-13 17:00:00


## 6️⃣ Residual monitoring (model drift)

In [6]:
print("=" * 70)
print("RESIDUAL MONITORING (Model Drift)")
print("=" * 70)

# Дефинирай defaults (за да не гръмне Cell 9, ако няма predictions)
drift_count = 0
baseline_mae = 0.0
drift_threshold = 0.0
rolling_mae = pd.Series(dtype=float)
rolling_mape = pd.Series(dtype=float)

if "actual" in pred_df.columns and "boosting_best" in pred_df.columns:
    pred_df["residual"] = pred_df["actual"] - pred_df["boosting_best"]
    pred_df["abs_residual"] = np.abs(pred_df["residual"])

    window = 168  # 1 седмица
    rolling_mae = pred_df["abs_residual"].rolling(window).mean()
    rolling_mape = (
        pred_df["abs_residual"] / pred_df["actual"].replace(0, np.nan)
    ).rolling(window).mean() * 100

    print(f"\nOverall:")
    print(f"  MAE:  {pred_df['abs_residual'].mean():.2f} MW")
    print(f"  MAPE: {(pred_df['abs_residual'] / pred_df['actual']).mean() * 100:.2f}%")

    baseline_window = 30 * 24
    baseline_mae = pred_df["abs_residual"].iloc[:baseline_window].mean()

    print(f"\nBaseline (first 30 days):")
    print(f"  MAE: {baseline_mae:.2f} MW")

    drift_threshold = baseline_mae * 1.5
    drift_mask = rolling_mae > drift_threshold
    drift_count = int(drift_mask.sum())

    print(f"\nDrift detection (threshold = 1.5x baseline = {drift_threshold:.1f} MW):")
    print(f"  Drift points: {drift_count} ({drift_count / len(pred_df) * 100:.1f}% от test)")

    if drift_count > 0:
        first_drift_idx = drift_mask.idxmax()
        print(f"  First drift at: {pred_df['timestamp'].iloc[first_drift_idx]}")

RESIDUAL MONITORING (Model Drift)

Overall:
  MAE:  29.25 MW
  MAPE: 2.95%

Baseline (first 30 days):
  MAE: 31.05 MW

Drift detection (threshold = 1.5x baseline = 46.6 MW):
  Drift points: 0 (0.0% от test)


## 7️⃣ Feature drift analysis

In [7]:
print("=" * 70)
print("FEATURE DRIFT ANALYSIS")
print("=" * 70)

train_early = train.iloc[:len(train)//2]
train_late = train.iloc[len(train)//2:]

feature_drift = {}
for col in ["consumption_mw", "temperature_c"]:
    psi_val = psi(train_early[col].values, train_late[col].values)
    ks_stat, ks_p = stats.ks_2samp(train_early[col].values, train_late[col].values)

    feature_drift[col] = {"psi": psi_val, "ks_stat": ks_stat, "ks_p": ks_p}

    print(f"\n{col}:")
    print(f"  PSI (early vs late train): {psi_val:.4f}")
    print(f"  KS stat: {ks_stat:.4f}, p = {ks_p:.4e}")

print("\n" + "=" * 70)
print("TEMPERATURE BY SEASON")
print("=" * 70)

df_season = df.copy()
df_season["month"] = df_season.index.month
df_season["season"] = df_season["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
})

season_stats = df_season.groupby("season")[["temperature_c", "consumption_mw"]].mean()
print(season_stats.to_string())

FEATURE DRIFT ANALYSIS

consumption_mw:
  PSI (early vs late train): 0.0057
  KS stat: 0.0321, p = 1.4569e-03

temperature_c:
  PSI (early vs late train): 0.1715
  KS stat: 0.1973, p = 9.9156e-120

TEMPERATURE BY SEASON
        temperature_c  consumption_mw
season                               
Autumn      14.627125      961.736471
Spring      15.168455      998.426469
Summer      26.681529     1278.514843
Winter       3.269761     1277.273212


## 8️⃣ Alert system (първоначален — Fix 3 го заменя)

In [8]:
print("=" * 70)
print("ALERT SYSTEM (initial — Fix 3 дава финален verdict)")
print("=" * 70)

thresholds = {
    "psi_consumption": {"warn": 0.1, "alert": 0.25},
    "psi_temperature": {"warn": 0.1, "alert": 0.25},
    "mae_multiplier": {"warn": 1.3, "alert": 1.5},
}

print("\nThresholds:")
for k, v in thresholds.items():
    print(f"  {k}: warn={v['warn']}, alert={v['alert']}")

alerts = []

psi_cons = psi_results.get("consumption_mw", 0)
if psi_cons >= thresholds["psi_consumption"]["alert"]:
    alerts.append(("🚨 ALERT", f"PSI consumption = {psi_cons:.4f}"))
elif psi_cons >= thresholds["psi_consumption"]["warn"]:
    alerts.append(("⚠️ WARN", f"PSI consumption = {psi_cons:.4f}"))
else:
    alerts.append(("✅ OK", f"PSI consumption = {psi_cons:.4f}"))

psi_temp = psi_results.get("temperature_c", 0)
if psi_temp >= thresholds["psi_temperature"]["alert"]:
    alerts.append(("🚨 ALERT", f"PSI temperature = {psi_temp:.4f}"))
elif psi_temp >= thresholds["psi_temperature"]["warn"]:
    alerts.append(("⚠️ WARN", f"PSI temperature = {psi_temp:.4f}"))
else:
    alerts.append(("✅ OK", f"PSI temperature = {psi_temp:.4f}"))

if "actual" in pred_df.columns and drift_count == 0 and baseline_mae > 0:
    mae_ratio = pred_df["abs_residual"].iloc[-30*24:].mean() / baseline_mae
    if mae_ratio >= thresholds["mae_multiplier"]["alert"]:
        alerts.append(("🚨 ALERT", f"MAE ratio = {mae_ratio:.2f}"))
    elif mae_ratio >= thresholds["mae_multiplier"]["warn"]:
        alerts.append(("⚠️ WARN", f"MAE ratio = {mae_ratio:.2f}"))
    else:
        alerts.append(("✅ OK", f"MAE ratio = {mae_ratio:.2f}"))

print("\nAlerts:")
for status, msg in alerts:
    print(f"  {status}: {msg}")

n_alerts = sum(1 for s, _ in alerts if "ALERT" in s)
n_warns = sum(1 for s, _ in alerts if "WARN" in s)

print("\nInitial recommendation:")
if n_alerts > 0:
    print("  🚨 RETRAIN NOW — significant drift detected (МОЖЕ ДА Е FALSE POSITIVE)")
elif n_warns > 0:
    print("  ⚠️ MONITOR — moderate drift")
else:
    print("  ✅ NO ACTION — model is stable")

ALERT SYSTEM (initial — Fix 3 дава финален verdict)

Thresholds:
  psi_consumption: warn=0.1, alert=0.25
  psi_temperature: warn=0.1, alert=0.25
  mae_multiplier: warn=1.3, alert=1.5

Alerts:
  🚨 ALERT: PSI consumption = 0.5318
  ✅ OK: PSI temperature = 0.0883
  ✅ OK: MAE ratio = 1.08

Initial recommendation:
  🚨 RETRAIN NOW — significant drift detected (МОЖЕ ДА Е FALSE POSITIVE)


## 9️⃣ Визуализации

In [9]:
fig, axes = plt.subplots(3, 2, figsize=(18, 14))

# (a) Consumption distribution — train vs test
axes[0, 0].hist(train["consumption_mw"], bins=50, alpha=0.5, label="Train", color="#3498db", density=True)
axes[0, 0].hist(test["consumption_mw"], bins=50, alpha=0.5, label="Test", color="#e74c3c", density=True)
axes[0, 0].set_title(f"Consumption distribution (PSI={psi_results.get('consumption_mw', 0):.3f})")
axes[0, 0].set_xlabel("MW")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# (b) Temperature distribution — train vs test
axes[0, 1].hist(train["temperature_c"], bins=50, alpha=0.5, label="Train", color="#3498db", density=True)
axes[0, 1].hist(test["temperature_c"], bins=50, alpha=0.5, label="Test", color="#e74c3c", density=True)
axes[0, 1].set_title(f"Temperature distribution (PSI={psi_results.get('temperature_c', 0):.3f})")
axes[0, 1].set_xlabel("°C")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# (c) Residuals over time
if "residual" in pred_df.columns:
    axes[1, 0].plot(pred_df["timestamp"], pred_df["residual"], color="#c0392b", lw=0.5, alpha=0.7)
    axes[1, 0].axhline(0, color="black", ls="--", alpha=0.5)
    axes[1, 0].set_title("Residuals over test period")
    axes[1, 0].set_ylabel("Residual (MW)")
    axes[1, 0].grid(alpha=0.3)

# (d) Rolling MAE
if "residual" in pred_df.columns and len(rolling_mae) > 0:
    axes[1, 1].plot(pred_df["timestamp"], rolling_mae, color="#9b59b6", lw=1.5, label="Rolling MAE (7d)")
    axes[1, 1].axhline(baseline_mae, color="green", ls="--", label=f"Baseline ({baseline_mae:.1f} MW)")
    axes[1, 1].axhline(drift_threshold, color="red", ls="--", label=f"Drift threshold ({drift_threshold:.1f} MW)")
    axes[1, 1].set_title("Rolling MAE over time")
    axes[1, 1].set_ylabel("MAE (MW)")
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

# (e) Temperature by season
if "season" in df_season.columns:
    seasons = ["Winter", "Spring", "Summer", "Autumn"]
    temp_data = [df_season[df_season["season"] == s]["temperature_c"].values for s in seasons]
    axes[2, 0].boxplot(temp_data, tick_labels=seasons)
    axes[2, 0].set_title("Temperature by season")
    axes[2, 0].set_ylabel("°C")
    axes[2, 0].grid(axis="y", alpha=0.3)

# (f) Consumption by season
if "season" in df_season.columns:
    cons_data = [df_season[df_season["season"] == s]["consumption_mw"].values for s in seasons]
    axes[2, 1].boxplot(cons_data, tick_labels=seasons)
    axes[2, 1].set_title("Consumption by season")
    axes[2, 1].set_ylabel("MW")
    axes[2, 1].grid(axis="y", alpha=0.3)

plt.suptitle("Drift Detection — Full Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07a_drift_analysis.png", bbox_inches="tight")
plt.show()
print(f"💾 Saved: {FIGURES_DIR / '07a_drift_analysis.png'}")

💾 Saved: D:\CUDA\Python\ClassicalML\GridForecast\notebooks\figures\07a_drift_analysis.png


## 🔟 Extended drift analysis (без запис на parquet — виж финалния cell)

In [10]:
print("=" * 70)
print("1. TRAIN vs TEST — BASIC STATS")
print("=" * 70)

for col in ["consumption_mw", "temperature_c"]:
    train_mean = train[col].mean()
    test_mean = test[col].mean()
    train_std = train[col].std()
    test_std = test[col].std()
    diff_pct = (test_mean - train_mean) / train_mean * 100
    print(f"\n{col}:")
    print(f"  Train: mean={train_mean:.2f}, std={train_std:.2f}")
    print(f"  Test:  mean={test_mean:.2f}, std={test_std:.2f}")
    print(f"  Δmean: {diff_pct:+.2f}%")
    print(f"  Δstd:  {(test_std / train_std - 1) * 100:+.2f}%")

print("\n" + "=" * 70)
print("2. MONTHLY BREAKDOWN")
print("=" * 70)

df_month = df.copy()
df_month["month"] = df_month.index.month
df_month["year"] = df_month.index.year
monthly_stats = df_month.groupby(["year", "month"])[["consumption_mw", "temperature_c"]].agg(["mean", "std", "count"])
print("\nMonthly stats:")
print(monthly_stats.round(2).to_string())

print("\n" + "=" * 70)
print("3. TRAIN / TEST OVERLAP (по месеци)")
print("=" * 70)

train_months = set(train.index.month)
test_months = set(test.index.month)
overlap = train_months & test_months

print(f"Train months: {sorted(train_months)}")
print(f"Test months:  {sorted(test_months)}")
print(f"Overlap:      {sorted(overlap)}")
print(f"Test-only:    {sorted(test_months - train_months)}")

if len(overlap) < len(test_months):
    print("\n⚠️ PROBLEM: Test has months NOT in train!")
    print("   → Моделът не е виждал тези сезони")
    print("   → Това е причина за висок PSI (false positive)")

print("\n" + "=" * 70)
print("4. CONSUMPTION BY HOUR — Train vs Test")
print("=" * 70)

train_hourly = train.groupby(train.index.hour)["consumption_mw"].mean()
test_hourly = test.groupby(test.index.hour)["consumption_mw"].mean()
hourly_compare = pd.DataFrame({
    "hour": range(24),
    "train_mean": train_hourly.values,
    "test_mean": test_hourly.values,
})
hourly_compare["diff"] = hourly_compare["test_mean"] - hourly_compare["train_mean"]
hourly_compare["diff_pct"] = hourly_compare["diff"] / hourly_compare["train_mean"] * 100
print(hourly_compare.round(2).to_string(index=False))

print("\n" + "=" * 70)
print("5. CONSUMPTION BY DOW — Train vs Test")
print("=" * 70)

train_dow = train.groupby(train.index.dayofweek)["consumption_mw"].mean()
test_dow = test.groupby(test.index.dayofweek)["consumption_mw"].mean()
dow_compare = pd.DataFrame({
    "dow": ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    "train_mean": train_dow.values,
    "test_mean": test_dow.values,
})
dow_compare["diff"] = dow_compare["test_mean"] - dow_compare["train_mean"]
dow_compare["diff_pct"] = dow_compare["diff"] / dow_compare["train_mean"] * 100
print(dow_compare.round(2).to_string(index=False))

1. TRAIN vs TEST — BASIC STATS

consumption_mw:
  Train: mean=1157.47, std=263.06
  Test:  mean=1013.28, std=172.48
  Δmean: -12.46%
  Δstd:  -34.43%

temperature_c:
  Train: mean=15.20, std=10.32
  Test:  mean=14.22, std=8.93
  Δmean: -6.43%
  Δstd:  -13.47%

2. MONTHLY BREAKDOWN

Monthly stats:
           consumption_mw               temperature_c            
                     mean     std count          mean   std count
year month                                                       
2023 1            1464.10  236.05   744          2.07  4.13   744
     2            1334.10  203.17   672          4.02  4.16   672
     3            1079.51  152.96   744          8.46  4.42   744
     4             920.45  142.50   720         15.16  4.51   720
     5             998.66  168.05   744         21.63  4.47   744
     6            1244.44  205.44   720         26.19  4.09   720
     7            1358.10  235.50   744         27.70  4.19   744
     8            1233.79  201.87   744   

In [11]:
print("\n" + "=" * 70)
print("6. TEMPERATURE DISTRIBUTION")
print("=" * 70)

print(f"Train temp: min={train['temperature_c'].min():.1f}, max={train['temperature_c'].max():.1f}")
print(f"Test temp:  min={test['temperature_c'].min():.1f}, max={test['temperature_c'].max():.1f}")

print(f"\nTrain temp percentiles:")
for p in [10, 25, 50, 75, 90]:
    print(f"  P{p}: {np.percentile(train['temperature_c'], p):.1f}")

print(f"\nTest temp percentiles:")
for p in [10, 25, 50, 75, 90]:
    print(f"  P{p}: {np.percentile(test['temperature_c'], p):.1f}")

print("\n" + "=" * 70)
print("7. PER-FEATURE PSI (detailed)")
print("=" * 70)

def psi_detailed(expected, actual, buckets=10):
    expected = np.asarray(expected)
    actual = np.asarray(actual)
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints)
    exp_counts, _ = np.histogram(expected, bins=breakpoints)
    act_counts, _ = np.histogram(actual, bins=breakpoints)
    exp_pct = (exp_counts + 0.0001) / (len(expected) + 0.0001 * len(breakpoints))
    act_pct = (act_counts + 0.0001) / (len(actual) + 0.0001 * len(breakpoints))
    psi_values = (act_pct - exp_pct) * np.log(act_pct / exp_pct)
    return np.sum(psi_values), breakpoints, exp_pct, act_pct, psi_values

psi_total, breaks, exp_pct, act_pct, psi_bins = psi_detailed(
    train["consumption_mw"].values,
    test["consumption_mw"].values,
)

print(f"\nConsumption PSI = {psi_total:.4f}")
print(f"\nPSI breakdown по bin:")
print(f"{'Bin range':>30s}  {'Train %':>10s}  {'Test %':>10s}  {'PSI contrib':>12s}")
for i in range(len(breaks) - 1):
    bin_range = f"[{breaks[i]:.0f}, {breaks[i+1]:.0f}]"
    print(f"{bin_range:>30s}  {exp_pct[i]*100:>9.2f}%  {act_pct[i]*100:>9.2f}%  {psi_bins[i]:>12.4f}")


6. TEMPERATURE DISTRIBUTION
Train temp: min=-8.7, max=38.3
Test temp:  min=-6.8, max=36.0

Train temp percentiles:
  P10: 1.3
  P25: 6.5
  P50: 15.4
  P75: 23.9
  P90: 28.9

Test temp percentiles:
  P10: 2.3
  P25: 7.2
  P50: 14.0
  P75: 21.4
  P90: 26.4

7. PER-FEATURE PSI (detailed)

Consumption PSI = 0.5318

PSI breakdown по bin:
                     Bin range     Train %      Test %   PSI contrib
                    [522, 859]      10.00%      17.32%        0.0402
                    [859, 931]      10.00%      17.78%        0.0448
                    [931, 988]      10.00%      12.44%        0.0053
                   [988, 1055]      10.00%      14.44%        0.0164
                  [1055, 1116]      10.00%      13.44%        0.0102
                  [1116, 1181]      10.00%       9.73%        0.0001
                  [1181, 1270]      10.00%       7.25%        0.0089
                  [1270, 1389]      10.00%       4.99%        0.0347
                  [1389, 1547]      10.00% 

In [12]:
fig, axes = plt.subplots(3, 2, figsize=(18, 14))

axes[0, 0].hist(train["consumption_mw"], bins=50, alpha=0.5, label="Train", color="#3498db", density=True)
axes[0, 0].hist(test["consumption_mw"], bins=50, alpha=0.5, label="Test", color="#e74c3c", density=True)
axes[0, 0].axvline(train["consumption_mw"].mean(), color="#3498db", ls="--", lw=2, label=f"Train mean={train['consumption_mw'].mean():.0f}")
axes[0, 0].axvline(test["consumption_mw"].mean(), color="#e74c3c", ls="--", lw=2, label=f"Test mean={test['consumption_mw'].mean():.0f}")
axes[0, 0].set_title("Consumption distribution — Train vs Test")
axes[0, 0].set_xlabel("MW")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].hist(train["temperature_c"], bins=50, alpha=0.5, label="Train", color="#3498db", density=True)
axes[0, 1].hist(test["temperature_c"], bins=50, alpha=0.5, label="Test", color="#e74c3c", density=True)
axes[0, 1].axvline(train["temperature_c"].mean(), color="#3498db", ls="--", lw=2, label=f"Train mean={train['temperature_c'].mean():.1f}")
axes[0, 1].axvline(test["temperature_c"].mean(), color="#e74c3c", ls="--", lw=2, label=f"Test mean={test['temperature_c'].mean():.1f}")
axes[0, 1].set_title("Temperature distribution — Train vs Test")
axes[0, 1].set_xlabel("°C")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

monthly = df_month.groupby(df_month.index.month)["consumption_mw"].mean()
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
axes[1, 0].bar(monthly.index, monthly.values, color="#9b59b6", edgecolor="black")
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(month_names)
axes[1, 0].set_title("Consumption by month (all data)")
axes[1, 0].set_ylabel("MW")
axes[1, 0].grid(alpha=0.3)

monthly_temp = df_month.groupby(df_month.index.month)["temperature_c"].mean()
axes[1, 1].bar(monthly_temp.index, monthly_temp.values, color="#e67e22", edgecolor="black")
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].set_xticklabels(month_names)
axes[1, 1].set_title("Temperature by month (all data)")
axes[1, 1].set_ylabel("°C")
axes[1, 1].grid(alpha=0.3)

axes[2, 0].plot(train_hourly.index, train_hourly.values, marker="o", color="#3498db", lw=2, label="Train")
axes[2, 0].plot(test_hourly.index, test_hourly.values, marker="s", color="#e74c3c", lw=2, label="Test")
axes[2, 0].set_title("Consumption by hour — Train vs Test")
axes[2, 0].set_xlabel("Hour")
axes[2, 0].set_ylabel("MW")
axes[2, 0].set_xticks(range(0, 24, 2))
axes[2, 0].legend()
axes[2, 0].grid(alpha=0.3)

bins_labels = [f"{breaks[i]:.0f}-{breaks[i+1]:.0f}" for i in range(len(breaks)-1)]
colors = ["#e74c3c" if x > 0.01 else "#2ecc71" for x in psi_bins]
axes[2, 1].bar(range(len(psi_bins)), psi_bins, color=colors, edgecolor="black")
axes[2, 1].set_xticks(range(len(psi_bins)))
axes[2, 1].set_xticklabels(bins_labels, rotation=45, ha="right")
axes[2, 1].set_title(f"PSI contribution by bin (total={psi_total:.3f})")
axes[2, 1].set_ylabel("PSI contribution")
axes[2, 1].grid(axis="y", alpha=0.3)

plt.suptitle("Extended Drift Analysis — Train vs Test", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07b_drift_extended.png", bbox_inches="tight")
plt.show()
print(f"\n💾 Saved: {FIGURES_DIR / '07b_drift_extended.png'}")


💾 Saved: D:\CUDA\Python\ClassicalML\GridForecast\notebooks\figures\07b_drift_extended.png


## 🔧 Fix 1: Seasonal PSI — сравни СЪЩИТЕ сезони

**Проблем:** Full PSI = 0.53 → false positive, защото train съдържа всички сезони, а test само есен/зима.

**Fix:** Сравни **същите месеци** (Oct-Dec 2023 vs Oct-Dec 2024).

In [13]:
print("=" * 70)
print("FIX 1: SEASONAL PSI")
print("=" * 70)

test_months = [10, 11, 12]

train_seasonal = train[train.index.month.isin(test_months)]
test_seasonal = test[test.index.month.isin(test_months)]

print(f"\nTrain Oct-Dec: {len(train_seasonal)} точки ({train_seasonal.index.min()} → {train_seasonal.index.max()})")
print(f"Test Oct-Dec:  {len(test_seasonal)} точки ({test_seasonal.index.min()} → {test_seasonal.index.max()})")

psi_seasonal = psi(train_seasonal["consumption_mw"].values, test_seasonal["consumption_mw"].values)
ks_stat_s, ks_p_s = stats.ks_2samp(train_seasonal["consumption_mw"].values, test_seasonal["consumption_mw"].values)
psi_temp_seasonal = psi(train_seasonal["temperature_c"].values, test_seasonal["temperature_c"].values)

print(f"\n📊 SEASONAL PSI (Oct-Dec comparison):")
print(f"   PSI: {psi_seasonal:.4f}")
if psi_seasonal < 0.1:
    print("   ✅ No drift — моделът работи!")
elif psi_seasonal < 0.25:
    print("   ⚠️ Moderate drift")
else:
    print("   🚨 Significant drift")

print(f"\n📊 SEASONAL KS test:")
print(f"   KS stat: {ks_stat_s:.4f}")
print(f"   p-value: {ks_p_s:.4e}")
if ks_p_s > 0.05:
    print("   ✅ No drift")
else:
    print("   🚨 Drift detected")

print(f"\n📊 SEASONAL PSI (temperature):")
print(f"   PSI: {psi_temp_seasonal:.4f}")

print(f"\n{'='*70}")
print("СРАВНЕНИЕ: Full vs Seasonal PSI")
print(f"{'='*70}")
print(f"  Full PSI (consumption):     {psi_results.get('consumption_mw', 0):.4f}  🚨")
print(f"  Seasonal PSI (consumption): {psi_seasonal:.4f}  {'✅' if psi_seasonal < 0.1 else '⚠️' if psi_seasonal < 0.25 else '🚨'}")
print(f"\n  → Full PSI е FALSE POSITIVE (сезонен артефакт)")

FIX 1: SEASONAL PSI

Train Oct-Dec: 2208 точки (2023-10-01 00:00:00 → 2023-12-31 23:00:00)
Test Oct-Dec:  2184 точки (2024-10-01 00:00:00 → 2024-12-30 23:00:00)

📊 SEASONAL PSI (Oct-Dec comparison):
   PSI: 0.0036
   ✅ No drift — моделът работи!

📊 SEASONAL KS test:
   KS stat: 0.0145
   p-value: 9.7170e-01
   ✅ No drift

📊 SEASONAL PSI (temperature):
   PSI: 0.0036

СРАВНЕНИЕ: Full vs Seasonal PSI
  Full PSI (consumption):     0.5318  🚨
  Seasonal PSI (consumption): 0.0036  ✅

  → Full PSI е FALSE POSITIVE (сезонен артефакт)


## 🔧 Fix 2: Hourly + DOW pattern comparison (normalized)

**Проблем:** Разликата в mean-а е сезонна (≈ -12%), но **формата** на profile-а може да е същата.

**Fix:** Нормализирай всеки profile и сравни **shape**-а (concept drift test).

In [14]:
print("=" * 70)
print("FIX 2: HOURLY PATTERN COMPARISON")
print("=" * 70)

train_hourly = train.groupby(train.index.hour)["consumption_mw"].mean()
test_hourly = test.groupby(test.index.hour)["consumption_mw"].mean()
train_norm = train_hourly / train_hourly.mean()
test_norm = test_hourly / test_hourly.mean()
diff = np.abs(train_norm - test_norm)
mean_diff_pct = float(diff.mean() * 100)
max_diff_pct = float(diff.max() * 100)

print(f"\n📊 Hourly pattern difference (normalized):")
print(f"   Mean diff: {mean_diff_pct:.2f}%")
print(f"   Max diff:  {max_diff_pct:.2f}% (at hour {diff.idxmax()})")
if mean_diff_pct < 5:
    print("   ✅ Patterns are SIMILAR — no concept drift")
elif mean_diff_pct < 10:
    print("   ⚠️ Moderate difference")
else:
    print("   🚨 Significant difference — concept drift")

train_dow = train.groupby(train.index.dayofweek)["consumption_mw"].mean()
test_dow = test.groupby(test.index.dayofweek)["consumption_mw"].mean()
train_dow_norm = train_dow / train_dow.mean()
test_dow_norm = test_dow / test_dow.mean()
dow_diff = np.abs(train_dow_norm - test_dow_norm)
dow_mean_pct = float(dow_diff.mean() * 100)

print(f"\n📊 DOW pattern difference (normalized):")
print(f"   Mean diff: {dow_mean_pct:.2f}%")
if dow_mean_pct < 5:
    print("   ✅ Patterns are SIMILAR — no concept drift")
else:
    print("   ⚠️ Some difference")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(train_norm.index, train_norm.values, marker="o", color="#3498db", lw=2, label="Train (normalized)")
axes[0].plot(test_norm.index, test_norm.values, marker="s", color="#e74c3c", lw=2, label="Test (normalized)")
axes[0].set_title(f"Hourly pattern (mean diff = {mean_diff_pct:.2f}%)")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Normalized consumption")
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(train_dow_norm.index, train_dow_norm.values, marker="o", color="#3498db", lw=2, label="Train")
axes[1].plot(test_dow_norm.index, test_dow_norm.values, marker="s", color="#e74c3c", lw=2, label="Test")
axes[1].set_title(f"DOW pattern (mean diff = {dow_mean_pct:.2f}%)")
axes[1].set_xlabel("Day of week")
axes[1].set_ylabel("Normalized consumption")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Pattern comparison — normalized hourly + DOW", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "07c_drift_patterns.png", bbox_inches="tight")
plt.show()
print(f"\n💾 Saved: {FIGURES_DIR / '07c_drift_patterns.png'}")

FIX 2: HOURLY PATTERN COMPARISON

📊 Hourly pattern difference (normalized):
   Mean diff: 1.04%
   Max diff:  2.08% (at hour 16)
   ✅ Patterns are SIMILAR — no concept drift

📊 DOW pattern difference (normalized):
   Mean diff: 0.84%
   ✅ Patterns are SIMILAR — no concept drift

💾 Saved: D:\CUDA\Python\ClassicalML\GridForecast\notebooks\figures\07c_drift_patterns.png


## 🔧 Fix 3: Final verdict — weighted (HIGH + CRIT checks)

**Проблем:** Full PSI + ADWIN дават false positives.

**Fix:** Даваме тежест на всеки check:
- `CRIT` — директно мери performance (residual drift)
- `HIGH` — сезонно-устойчиви метрики (seasonal PSI, pattern diff)
- `LOW` / `IGNORE` — false positives (full PSI, ADWIN при seasonal data)

In [15]:
print("=" * 70)
print("FIX 3: FINAL VERDICT (Weighted)")
print("=" * 70)

checks = [
    # name, value, threshold, status, weight, comment
    ("Full PSI",              psi_results.get("consumption_mw", 0), 0.25, None, "LOW",  "False positive (seasonal)"),
    ("Seasonal PSI",          psi_seasonal,                          0.25, None, "HIGH", "Най-важен за seasonal data"),
    ("Hourly pattern diff %", mean_diff_pct,                         5.00, None, "HIGH", "Проверява concept drift"),
    ("DOW pattern diff %",    dow_mean_pct,                          5.00, None, "HIGH", "Проверява concept drift"),
    ("Residual drift points", drift_count,                           0,    None, "CRIT", "Директно мери performance"),
    ("ADWIN drift points",    len(drift_points),                     5,    None, "LOW",  "False positive при seasonal"),
]

checks_with_status = []
for name, val, threshold, _, weight, comment in checks:
    if name in ["Full PSI", "ADWIN drift points"]:
        status = "🔕 IGNORE"
    else:
        if val <= threshold:
            status = "✅ OK"
        elif val <= threshold * 1.5:
            status = "⚠️ WARN"
        else:
            status = "🚨 ALERT"
    checks_with_status.append((name, val, threshold, status, weight, comment))

print(f"\n{'Check':<25} {'Value':>10} {'Threshold':>10} {'Weight':>6} {'Status':>10}  Comment")
print("=" * 100)
for name, val, threshold, status, weight, comment in checks_with_status:
    print(f"{name:<25} {val:>10.4f} {threshold:>10.2f} {weight:>6} {status:>10}  {comment}")

critical_checks = [(n, v, t, s) for n, v, t, s, w, _ in checks_with_status if w in ["HIGH", "CRIT"]]
n_alert = sum(1 for _, _, _, s in critical_checks if "ALERT" in s)
n_warn = sum(1 for _, _, _, s in critical_checks if "WARN" in s)
n_ok = sum(1 for _, _, _, s in critical_checks if "OK" in s)

print(f"\n{'='*70}")
print("WEIGHTED VERDICT (само HIGH + CRIT checks)")
print(f"{'='*70}")
print(f"  🚨 ALERT: {n_alert}")
print(f"  ⚠️ WARN:  {n_warn}")
print(f"  ✅ OK:    {n_ok}")
print(f"  🔕 IGNORE: {len(checks) - len(critical_checks)} (false positives)")

if n_alert > 0:
    final_verdict = "🚨 RETRAIN NOW"
elif n_warn > 0:
    final_verdict = "⚠️ MONITOR — schedule retraining"
else:
    final_verdict = "✅ NO ACTION — model is STABLE"

print(f"\n{'='*70}")
print("FINAL RECOMMENDATION")
print(f"{'='*70}")
print(f"  {final_verdict}")
if n_alert == 0 and n_warn == 0:
    print("     Recommendation: monthly monitoring + quarterly retraining")

print(f"\n{'='*70}")
print("WHY Full PSI AND ADWIN ARE FALSE POSITIVES")
print(f"{'='*70}")
print(f"\nFull PSI ({psi_results.get('consumption_mw', 0):.4f}):")
print(f"  - Train: Jan 2023 - Sep 2024 (ALL seasons)")
print(f"  - Test:  Oct-Dec 2024 (Autumn/Winter only)")
print(f"  - Δmean = -12% → seasonal effect, not drift")
print(f"  - Seasonal PSI = {psi_seasonal:.4f} → confirms NO real drift")
print(f"\nADWIN ({len(drift_points)} points):")
print(f"  - Работи на residuals, не на data")
print(f"  - Residuals имат сезонни pattern-и → false positives")
print(f"  - delta=0.01 е твърде строг")
print(f"  - Residual monitoring = {drift_count} → confirms NO real drift")

print(f"\n{'='*70}")
print("🎯 FINAL VERDICT")
print(f"{'='*70}")
print(f"  Data drift:     {'✅ NONE (seasonal artifact)' if psi_seasonal < 0.1 else '⚠️ CHECK'}")
print(f"  Pattern drift:  {'✅ NONE (same shape)' if mean_diff_pct < 5 and dow_mean_pct < 5 else '⚠️ CHECK'}")
print(f"  Model drift:    {'✅ NONE (residual monitoring OK)' if drift_count == 0 else '🚨 CHECK'}")
print()
print(f"  → {final_verdict}")

FIX 3: FINAL VERDICT (Weighted)

Check                          Value  Threshold Weight     Status  Comment
Full PSI                      0.5318       0.25    LOW   🔕 IGNORE  False positive (seasonal)
Seasonal PSI                  0.0036       0.25   HIGH       ✅ OK  Най-важен за seasonal data
Hourly pattern diff %         1.0391       5.00   HIGH       ✅ OK  Проверява concept drift
DOW pattern diff %            0.8387       5.00   HIGH       ✅ OK  Проверява concept drift
Residual drift points         0.0000       0.00   CRIT       ✅ OK  Директно мери performance
ADWIN drift points           25.0000       5.00    LOW   🔕 IGNORE  False positive при seasonal

WEIGHTED VERDICT (само HIGH + CRIT checks)
  🚨 ALERT: 0
  ⚠️ WARN:  0
  ✅ OK:    4
  🔕 IGNORE: 2 (false positives)

FINAL RECOMMENDATION
  ✅ NO ACTION — model is STABLE
     Recommendation: monthly monitoring + quarterly retraining

WHY Full PSI AND ADWIN ARE FALSE POSITIVES

Full PSI (0.5318):
  - Train: Jan 2023 - Sep 2024 (ALL se

## 💾 Финален запис — `drift_summary.parquet`

⚠️ **Важно:** Този cell е **последен**, за да запише **правилните** стойности (seasonal PSI, pattern diff, weighted verdict), а не старите false positives.

Файлът се чете от Фаза 2 API (`/model-info` endpoint).

In [16]:
# ============================================================
# Финален запис: drift_summary.parquet (ASCII status за API)
# ============================================================
# ВАЖНО: Записваме ASCII стойности (не emoji/кирилица), за да
# избегнем mojibake при pandas.read_parquet() в Python 3.11.
# Dashboard-ът/клиентът ще map-ва ASCII → emoji на frontend-а.
# ============================================================

# Mapping: emoji/кирилица → ASCII код
STATUS_MAP = {
    "🚨 Drift": "DRIFT",
    "✅ OK": "OK",
    "⚠️ WARN": "WARN",
    "⚠️ Moderate": "MODERATE",
    "🔕 IGNORE": "IGNORE",
    "🚨 RETRAIN NOW": "RETRAIN_NOW",
    "⚠️ MONITOR — schedule retraining": "MONITOR",
    "✅ NO ACTION — model is STABLE": "NO_ACTION_STABLE",
}


def normalize_status(s: str) -> str:
    """Map-ва emoji/кирилица status към ASCII код."""
    s = str(s).strip()
    # Опитай точно съвпадение
    if s in STATUS_MAP:
        return STATUS_MAP[s]
    # Fallback: substring matching
    if "DRIFT" in s.upper() or "DRIFT" in s:
        return "DRIFT"
    if "OK" in s.upper():
        return "OK"
    if "WARN" in s.upper():
        return "WARN"
    if "MODERATE" in s.upper():
        return "MODERATE"
    if "IGNORE" in s.upper():
        return "IGNORE"
    if "RETRAIN" in s.upper():
        return "RETRAIN_NOW"
    if "MONITOR" in s.upper():
        return "MONITOR"
    if "STABLE" in s.upper() or "NO ACTION" in s.upper():
        return "NO_ACTION_STABLE"
    return s


def normalize_note(s: str) -> str:
    """Заменя кирилица в note с английски."""
    s = str(s).strip()
    replacements = {
        "при seasonal residuals": "for seasonal residuals",
        "при seasonal": "for seasonal",
        "Най-важен за seasonal data": "most important for seasonal data",
        "Проверява concept drift": "checks concept drift",
        "Директно мери performance": "directly measures performance",
        "False positive (seasonal)": "False positive (seasonal)",
        "false positive (seasonal)": "false positive (seasonal)",
    }
    for bg, en in replacements.items():
        s = s.replace(bg, en)
    return s


summary = pd.DataFrame([
    {
        "method": "PSI (consumption, full)",
        "value": psi_results.get("consumption_mw", 0),
        "threshold": 0.25,
        "status": "DRIFT" if psi_results.get("consumption_mw", 0) >= 0.25 else "OK",
        "weight": "LOW",
        "note": "false positive (seasonal)",
    },
    {
        "method": "PSI (consumption, seasonal Oct-Dec)",
        "value": psi_seasonal,
        "threshold": 0.25,
        "status": "DRIFT" if psi_seasonal >= 0.25 else ("MODERATE" if psi_seasonal >= 0.1 else "OK"),
        "weight": "HIGH",
        "note": "real metric for seasonal data",
    },
    {
        "method": "PSI (temperature, full)",
        "value": psi_results.get("temperature_c", 0),
        "threshold": 0.25,
        "status": "DRIFT" if psi_results.get("temperature_c", 0) >= 0.25 else "OK",
        "weight": "LOW",
        "note": "",
    },
    {
        "method": "PSI (temperature, seasonal Oct-Dec)",
        "value": psi_temp_seasonal,
        "threshold": 0.25,
        "status": "DRIFT" if psi_temp_seasonal >= 0.25 else ("MODERATE" if psi_temp_seasonal >= 0.1 else "OK"),
        "weight": "HIGH",
        "note": "",
    },
    {
        "method": "KS (consumption, full)",
        "value": ks_results.get("consumption_mw", {}).get("p", 1.0),
        "threshold": 0.05,
        "status": "DRIFT" if ks_results.get("consumption_mw", {}).get("p", 1.0) < 0.05 else "OK",
        "weight": "LOW",
        "note": "p-value; p<0.05 = drift",
    },
    {
        "method": "KS (consumption, seasonal Oct-Dec)",
        "value": ks_p_s,
        "threshold": 0.05,
        "status": "DRIFT" if ks_p_s < 0.05 else "OK",
        "weight": "HIGH",
        "note": "p-value; p<0.05 = drift",
    },
    {
        "method": "KS (temperature, full)",
        "value": ks_results.get("temperature_c", {}).get("p", 1.0),
        "threshold": 0.05,
        "status": "DRIFT" if ks_results.get("temperature_c", {}).get("p", 1.0) < 0.05 else "OK",
        "weight": "LOW",
        "note": "p-value; p<0.05 = drift",
    },
    {
        "method": "Hourly pattern diff %",
        "value": mean_diff_pct,
        "threshold": 5.0,
        "status": "DRIFT" if mean_diff_pct >= 5 else "OK",
        "weight": "HIGH",
        "note": "concept drift; normalized profiles",
    },
    {
        "method": "DOW pattern diff %",
        "value": dow_mean_pct,
        "threshold": 5.0,
        "status": "DRIFT" if dow_mean_pct >= 5 else "OK",
        "weight": "HIGH",
        "note": "concept drift; normalized profiles",
    },
    {
        "method": "Residual drift points",
        "value": drift_count,
        "threshold": 0,
        "status": "DRIFT" if drift_count > 0 else "OK",
        "weight": "CRIT",
        "note": "MAE > 1.5x baseline",
    },
    {
        "method": "ADWIN drift points",
        "value": len(drift_points),
        "threshold": 5,
        "status": "IGNORE",
        "weight": "LOW",
        "note": "false positive for seasonal residuals",
    },
    {
        "method": "FINAL VERDICT",
        "value": float(n_alert * 2 + n_warn),
        "threshold": 0.0,
        "status": normalize_status(final_verdict),
        "weight": "OVERALL",
        "note": f"alerts={n_alert}, warns={n_warn}, ok={n_ok}",
    },
])

# Допълнително: нормализирай status/note колоните (за сигурност)
summary["status"] = summary["status"].apply(normalize_status)
summary["note"] = summary["note"].apply(normalize_note)

print(summary.to_string(index=False))

# Запиши с ASCII (без emoji/кирилица)
summary_path = MODELS_DIR / "drift_summary.parquet"
summary.to_parquet(summary_path, compression="snappy")
print(f"\n💾 Saved: {summary_path}")

# Верификация: провери, че няма non-ASCII в status/note
non_ascii = summary[
    summary["status"].apply(lambda s: any(ord(c) > 127 for c in str(s))) |
    summary["note"].apply(lambda s: any(ord(c) > 127 for c in str(s)))
]
if len(non_ascii) == 0:
    print("✅ No non-ASCII characters in status/note")
else:
    print(f"⚠️ Warning: {len(non_ascii)} rows contain non-ASCII:")
    print(non_ascii[["method", "status", "note"]])

                             method         value  threshold           status  weight                                  note
            PSI (consumption, full)  5.317848e-01       0.25            DRIFT     LOW             false positive (seasonal)
PSI (consumption, seasonal Oct-Dec)  3.647870e-03       0.25               OK    HIGH         real metric for seasonal data
            PSI (temperature, full)  8.833236e-02       0.25               OK     LOW                                      
PSI (temperature, seasonal Oct-Dec)  3.615337e-03       0.25               OK    HIGH                                      
             KS (consumption, full) 6.244955e-165       0.05            DRIFT     LOW               p-value; p<0.05 = drift
 KS (consumption, seasonal Oct-Dec)  9.717009e-01       0.05               OK    HIGH               p-value; p<0.05 = drift
             KS (temperature, full)  2.266941e-21       0.05            DRIFT     LOW               p-value; p<0.05 = drift
        

## ✅ Заключение

In [17]:
print("=" * 70)
print("ЗАКЛЮЧЕНИЕ (07_drift.ipynb)")
print("=" * 70)
print()
print("Drift detection методи:")
print(f"  1. PSI consumption (full):     {psi_results.get('consumption_mw', 0):.4f}  {'🚨' if psi_results.get('consumption_mw', 0) >= 0.25 else '✅'}")
print(f"  2. PSI consumption (seasonal): {psi_seasonal:.4f}  {'🚨' if psi_seasonal >= 0.25 else '✅'}")
print(f"  3. PSI temperature (full):     {psi_results.get('temperature_c', 0):.4f}")
print(f"  4. PSI temperature (seasonal): {psi_temp_seasonal:.4f}")
print(f"  5. Hourly pattern diff:        {mean_diff_pct:.2f}%  {'🚨' if mean_diff_pct >= 5 else '✅'}")
print(f"  6. DOW pattern diff:           {dow_mean_pct:.2f}%  {'🚨' if dow_mean_pct >= 5 else '✅'}")
print(f"  7. Residual drift points:      {drift_count}  {'🚨' if drift_count > 0 else '✅'}")
print(f"  8. ADWIN points:               {len(drift_points)}  (🔕 false positive)")
print()
print(f"Weighted verdict: {final_verdict}")
print(f"  alerts={n_alert}, warns={n_warn}, ok={n_ok}")
print()
print("Изводи:")
print("  - Full PSI/KS са false positives при seasonal data → IGNORE")
print("  - Seasonal PSI + pattern diff са истинските drift метрики")
print("  - Residual monitoring е най-прост за production")
print("  - Threshold-и: PSI > 0.25 (alert), MAE > 1.5x baseline (alert)")
print("  - При drift → retrain (виж 08_retraining.ipynb)")
print()
print(f"💾 drift_summary.parquet записан в {MODELS_DIR}")
print("=" * 70)

ЗАКЛЮЧЕНИЕ (07_drift.ipynb)

Drift detection методи:
  1. PSI consumption (full):     0.5318  🚨
  2. PSI consumption (seasonal): 0.0036  ✅
  3. PSI temperature (full):     0.0883
  4. PSI temperature (seasonal): 0.0036
  5. Hourly pattern diff:        1.04%  ✅
  6. DOW pattern diff:           0.84%  ✅
  7. Residual drift points:      0  ✅
  8. ADWIN points:               25  (🔕 false positive)

Weighted verdict: ✅ NO ACTION — model is STABLE
  alerts=0, warns=0, ok=4

Изводи:
  - Full PSI/KS са false positives при seasonal data → IGNORE
  - Seasonal PSI + pattern diff са истинските drift метрики
  - Residual monitoring е най-прост за production
  - Threshold-и: PSI > 0.25 (alert), MAE > 1.5x baseline (alert)
  - При drift → retrain (виж 08_retraining.ipynb)

💾 drift_summary.parquet записан в D:\CUDA\Python\ClassicalML\GridForecast\models
